# Faster R-CNN seed 0 · Kaggle 결과 분석

**실행 목적:** 완료한 **FT / 3e-4 / seed 0**의 저장 자료를 분석한다. 학습 루프·optimizer·역전파를 실행하지 않는다.
best·last 체크포인트에서는 가중치와 정규화 버퍼를 보존한 채 사후 train/validation loss만 측정한다.

**실행 순서:** 아래 **셀 1 → 8**을 차례대로 실행한다. `Run All`로 100 epoch 학습이 시작되지 않는다.
셀 6까지는 저장 자료 분석, 셀 7은 두 체크포인트 손실 재측정, 셀 8은 결과 압축·다운로드다.

**Kaggle 준비:** 새 Notebook → File / Import Notebook으로 이 파일 업로드 → Add Input에서 자료 추가.
손실 측정에는 Settings / Accelerator에서 사용 가능한 GPU를 선택한다. 설치를 위해 Internet을 켠다.
GPU가 비활성화된 계정은 Kaggle의 계정 인증·남은 GPU 할당량을 확인한다.

**업로드할 자료(Private Dataset):**
- Drive의 `cj_faster_e100_s0_fix_ft_lr3e-4__...` 실행 폴더 전체. `config.json`, `result.json`, `metrics.csv`,
  `class_map.csv`, `best.pt`, `last.pt`, `validation_predictions.pt`를 포함한다. 원래 `analysis/`도 보존해도 된다.
- 원래 이미지·annotation JSON·`split.csv`·실제로 사용했던 수정표. 기존 폴더 구조와 내용을 유지한다.
  자료가 이미 경진대회 입력에 있더라도 팀 수정표와 고정 분할은 따로 연결해야 한다.
- `.env`, W&B API 키 등은 업로드하지 않는다. 이 분석 노트북은 W&B 로그인을 요구하지 않는다.

**환경:** 팀의 `.python-version`, `pyproject.toml`, `uv.lock` 스냅샷을 이 파일에 포함했다.
uv로 Python 3.12의 별도 환경을 만들고 그 Python으로 분석 코드를 실행한다. Kaggle 기본 커널 재시작은 필요 없다.
패키지 환경은 `/tmp`에 두고 보고서만 `/kaggle/working`에 저장한다.
원래 학습의 잠금 식별자·패키지 버전과 대조하며, 맞지 않을 때 임의로 버전을 낮추지 않는다.
팀 CUDA 13 빌드가 Kaggle 드라이버와 호환되지 않으면 CPU 저장 자료 분석은 가능하지만 GPU 손실 측정은 중단한다.

**해석 범위:** 과거 epoch별 val loss는 새로 만들어내지 않는다. best·last의 사후 loss는 따로 표·그래프로 표시한다.
같은 작은 batch·측정 seed로 train과 val을 평가해 비교하며, 학습 당시 batch 16 loss와 동일한 수치로 간주하지 않는다.
두 체크포인트의 epoch가 같으면 한 시점으로 해석한다. 진단 precision/recall/F1은 mAP와 별개다.

[Kaggle 문서](https://www.kaggle.com/docs/notebooks) ·
[CUDA 드라이버 호환성](https://docs.nvidia.com/deploy/cuda-compatibility/minor-version-compatibility.html) ·
[torchvision Faster R-CNN](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html)


## 1. 입력·Kaggle 환경 준비

### 셀 1 · 업로드 경로·분석 조건

일치하는 seed 0 폴더가 하나면 자동 선택한다. 여러 개라면 오른쪽 Input에서 경로를 복사해 `RUN_DIR`에 입력한다.
데이터는 `data/raw/train_images`, `data/raw/train_annotations`, `data/processed/split.csv` 구조를 자동 탐색한다.
수정표는 업로드한 `label_fixes.csv` 또는 `jaeone_label_fixes.csv`를 사용하며, 후보가 여러 개면 명시한다.
자동 탐색에 실패하면 아래 네 경로를 입력한다. 이미지는 손실 계산과 오류 이미지 재시각화에 필요하다.

`LOSS_BATCH_SIZE=1`은 **현재 사후 측정**의 설정이다. 원래 학습 batch 16을 수정하지 않는다.
기본값은 best·last 각각 train·val 전체를 평가하는 총 네 번의 순회다. 시간 절약이 필요하면 `LOSS_SPLITS=["val"]`로 지정한다.


In [ ]:
import json
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUN_DIR = None
DATA_ROOT = None
IMAGE_DIR = None
ANNOTATION_DIR = None
SPLIT_PATH = None
FIXES_PATH = None
# 학습 당시 환경 파일을 별도로 올린 경우만 지정. 기본값은 내장 스냅샷.
TEAM_REFERENCE_DIR = None

LOSS_DEVICE = "cuda"
LOSS_BATCH_SIZE = 1
LOSS_MEASUREMENT_SEED = 0
LOSS_SPLITS = ["train", "val"]
DIAGNOSTIC_IOU = 0.75
DIAGNOSTIC_SCORE = 0.30
PREVIEW_COUNT = 6
OUTPUT_ROOT = WORK_ROOT / "cj_faster_seed0_analysis"

if not INPUT_ROOT.is_dir():
    raise RuntimeError("Kaggle Notebook에서 실행하세요. /kaggle/input에 실행 폴더를 Add Input으로 연결해야 합니다.")

if RUN_DIR is None:
    candidates = []
    for path in sorted(INPUT_ROOT.rglob("config.json")):
        try:
            item = json.loads(path.read_text())
            spec = item.get("spec", {})
            if spec.get("seed") == 0 and spec.get("mode") == "ft" and abs(float(spec.get("lr", -1)) - 3e-4) < 1e-12:
                candidates.append(path.parent)
        except (ValueError, OSError, TypeError):
            continue
    if len(candidates) != 1:
        print("실행 폴더 후보:", *map(str, candidates), sep="\n")
        raise RuntimeError("분석할 실행을 하나로 정할 수 없습니다. RUN_DIR에 같은 실행의 config.json이 있는 폴더를 입력하세요.")
    RUN_DIR = candidates[0]
RUN_DIR = Path(RUN_DIR)
if not (RUN_DIR / "config.json").is_file():
    raise FileNotFoundError("RUN_DIR에 config.json이 없습니다. 파일이 아닌 실행 폴더의 경로를 지정하세요.")
SOURCE_MANIFEST = json.loads((RUN_DIR / "config.json").read_text())
OUTPUT_DIR = OUTPUT_ROOT / (SOURCE_MANIFEST["run_id"] + "__" + SOURCE_MANIFEST["signature"][:12])

if DATA_ROOT is None:
    roots = [p.parent.parent for p in INPUT_ROOT.rglob("split.csv")
             if p.parent.name == "processed" and (p.parent.parent / "raw/train_images").is_dir()
             and (p.parent.parent / "raw/train_annotations").is_dir()]
    if len(roots) == 1:
        DATA_ROOT = roots[0]
if DATA_ROOT is not None:
    DATA_ROOT = Path(DATA_ROOT)
    IMAGE_DIR = IMAGE_DIR or DATA_ROOT / "raw/train_images"
    ANNOTATION_DIR = ANNOTATION_DIR or DATA_ROOT / "raw/train_annotations"
    SPLIT_PATH = SPLIT_PATH or DATA_ROOT / "processed/split.csv"
    if FIXES_PATH is None and (DATA_ROOT / "processed/label_fixes.csv").is_file():
        FIXES_PATH = DATA_ROOT / "processed/label_fixes.csv"
if FIXES_PATH is None:
    fixes = sorted({p for name in ["label_fixes.csv", "jaeone_label_fixes.csv"] for p in INPUT_ROOT.rglob(name)})
    if len(fixes) == 1:
        FIXES_PATH = fixes[0]

for name, value in [("RUN_DIR", RUN_DIR), ("IMAGE_DIR", IMAGE_DIR), ("ANNOTATION_DIR", ANNOTATION_DIR),
                    ("SPLIT_PATH", SPLIT_PATH), ("FIXES_PATH", FIXES_PATH)]:
    print(name, "=", value, "| exists:", Path(value).exists() if value is not None else False)
print("분석 출력:", OUTPUT_DIR)
if not all(v is not None and Path(v).exists() for v in [IMAGE_DIR, ANNOTATION_DIR, SPLIT_PATH, FIXES_PATH]):
    print("원본 데이터 경로가 부족합니다. 셀 6의 저장 지표 분석은 가능하지만 셀 7 손실 측정 전에는 위 경로를 모두 지정해야 합니다.")


### 셀 2 · 팀 고정 환경 설치

최초 패키지 다운로드는 시간이 걸릴 수 있다. 설치 진행이 출력된다.
분석 전용 Python을 사용하므로 **설치 후 커널 재시작 없이 셀 3으로 진행**한다.
내장 스냅샷에는 환경 파일 세 개만 있고 비밀키·데이터는 없다.


In [ ]:
TEAM_ENVIRONMENT_B64 = '...'

"""노트북에 포함된 팀 잠금 파일로 독립 분석 환경을 준비한다."""

import base64
import hashlib
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def stream_command(arguments, cwd=None, env=None):
    """설치·분석 진행 상황을 숨기지 않고 출력한다.

    Args:
        arguments (list): 실행 인수.
        cwd (str | Path | None): 작업 폴더.
        env (dict | None): 자식 프로세스 환경.
    """
    with subprocess.Popen(arguments, cwd=cwd, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError("작업이 완료되지 않았습니다. 바로 위 오류를 해결한 뒤 같은 셀을 다시 실행하세요.")


if not RUN_DIR.is_dir():
    raise FileNotFoundError("실행 폴더가 없습니다. 셀 1에서 RUN_DIR을 지정하세요.")
manifest_for_setup = json.loads((RUN_DIR / "config.json").read_text())
expected_lock_hash = manifest_for_setup["recipe"]["runtime"].get("uv_lock_sha256")
if not expected_lock_hash:
    raise RuntimeError("원래 학습의 uv.lock 식별자가 없습니다. config.json의 runtime 기록을 확인하세요.")

# 패키지 수 GB가 Kaggle 결과에 함께 저장되지 않도록 임시 영역에 둔다.
ENV_ROOT = Path("/tmp/cj_faster_seed0_environment") / expected_lock_hash[:12]
REFERENCE_ROOT = ENV_ROOT / "references"
REFERENCE_ROOT.mkdir(parents=True, exist_ok=True)
reference_names = (".python-version", "pyproject.toml", "uv.lock")
if TEAM_REFERENCE_DIR is not None:
    references = {name: (Path(TEAM_REFERENCE_DIR) / name).read_bytes() for name in reference_names}
else:
    # 환경 파일 세 개만 들어 있으며 .env·키·프로젝트 데이터는 포함하지 않는다.
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(TEAM_ENVIRONMENT_B64))) as archive:
        if set(archive.namelist()) != set(reference_names):
            raise RuntimeError("내장 환경 자료가 잘못됐습니다. 노트북 원본을 확인하세요.")
        references = {name: archive.read(name) for name in reference_names}
if hashlib.sha256(references["uv.lock"]).hexdigest() != expected_lock_hash:
    raise RuntimeError("내장 uv.lock과 원래 학습의 잠금 식별자가 다릅니다. 당시 환경 파일 세 개를 업로드하고 TEAM_REFERENCE_DIR을 지정하세요. 버전을 임의로 변경하지 않습니다.")
if references[".python-version"].decode().strip() != "3.12":
    raise RuntimeError("팀 Python 기준이 3.12가 아닙니다. 원래 환경 파일을 확인하세요.")
for name, content in references.items():
    destination = REFERENCE_ROOT / name
    if not destination.exists() or destination.read_bytes() != content:
        destination.write_bytes(content)

uv_binary = shutil.which("uv")
if uv_binary is None:
    local_uv = Path.home() / ".local/bin/uv"
    if not local_uv.exists():
        print("공식 uv 설치 프로그램을 준비합니다. Kaggle Internet을 켜 두세요.", flush=True)
        stream_command(["bash", "-o", "pipefail", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"])
    uv_binary = str(local_uv)

ANALYSIS_VENV = ENV_ROOT / "venv"
print("팀 잠금 환경 설치 중… 최초 실행에는 CUDA 패키지 다운로드 시간이 필요합니다.", flush=True)
stream_command([uv_binary, "sync", "--locked", "--no-dev", "--no-install-project", "--python", "3.12"],
               cwd=REFERENCE_ROOT, env={**os.environ, "UV_PROJECT_ENVIRONMENT": str(ANALYSIS_VENV),
                                        "UV_CACHE_DIR": str(ENV_ROOT / "uv_cache")})
for name, content in references.items():
    if (REFERENCE_ROOT / name).read_bytes() != content:
        raise RuntimeError("환경 기준 파일이 변경됐습니다. 설치를 중단하고 잠금 환경을 확인하세요.")
ANALYSIS_PYTHON = ANALYSIS_VENV / "bin/python"
print("분석용 Python:", ANALYSIS_PYTHON)
print("현재 Kaggle 커널은 교체하지 않습니다. Restart 없이 다음 셀로 진행하세요.")


## 2. 내장 함수 준비

### 셀 3 · 데이터·평가·분석 함수

필요한 구현을 이 노트북 안에 포함했다. 별도 공용 Python 파일을 GitHub에 올릴 필요가 없다.
문자열로 정의한 코드는 셀 4에서 팀 환경의 자식 프로세스로 전달한다. 기존 학습 함수를 포함하지 않는다.


In [ ]:
INLINE_ANALYSIS_BLOCKS = {}

INLINE_ANALYSIS_BLOCKS['helpers'] = r'''
from PIL import Image, ImageDraw, ImageFont

def xyxy_to_xywh(bbox):
    """Ultralytics 추론 결과의 xyxy bbox를 xywh로 바꾼다.

    Args:
        bbox (list[float]): [좌상단 x, 좌상단 y, 우하단 x, 우하단 y]. 원본 픽셀.

    Returns:
        list[float]: [좌상단 x, 좌상단 y, 너비, 높이]. 원본 픽셀.
    """
    x1, y1, x2, y2 = bbox
    return [x1, y1, x2 - x1, y2 - y1]

def draw_boxes(image, boxes, labels=None, color="red", width=4):
    """이미지에 xywh bbox를 그려 새 PIL 이미지로 돌려준다.

    기본 글꼴은 한글을 그리지 못하므로 labels에는 category_id 같은 숫자·영문을 넣는다.

    Args:
        image (str | Path | PIL.Image.Image): 이미지 경로나 PIL 이미지.
        boxes (list[list[float]]): 그릴 bbox 목록. 각각 [좌상단 x, 좌상단 y, 너비, 높이], 원본 픽셀.
        labels (list | None): bbox마다 위에 적을 글자. boxes와 순서가 같다. None이면 적지 않는다.
        color (str): 선과 글자의 색.
        width (int): 선 두께(픽셀).

    Returns:
        PIL.Image.Image: bbox를 그린 RGB 이미지.
    """
    image = Image.open(image) if not isinstance(image, Image.Image) else image
    image = image.convert("RGB")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=28)
    for i, (x, y, w, h) in enumerate(boxes):
        draw.rectangle([x, y, x + w, y + h], outline=color, width=width)
        if labels:
            draw.text((x, max(0, y - 32)), str(labels[i]), fill=color, font=font)
    return image
'''

INLINE_ANALYSIS_BLOCKS['data'] = r'''
"""알약 검출의 고정 분할·수정 라벨·원본 좌표를 관리한다."""

import ast
import hashlib
import json
import os
import shutil
import tempfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.ops import box_convert
from torchvision.transforms.functional import to_tensor


def file_sha256(path):
    """파일 내용을 해시한다.

    Args:
        path (str | Path): 검사할 파일.

    Returns:
        str: SHA-256.
    """
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(value):
    """JSON으로 표현 가능한 설정의 식별자를 만든다.

    Args:
        value (object): 해시할 값.

    Returns:
        str: SHA-256.
    """
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode()).hexdigest()


def write_json(path, value):
    """임시 파일을 완성한 뒤 원자적으로 JSON을 저장한다.

    Args:
        path (str | Path): 저장 위치.
        value (object): JSON 값.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=path.name, delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(text)
    os.replace(temporary, path)


def _bbox(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    array = np.asarray(value, dtype=float)
    if array.shape != (4,) or not np.isfinite(array).all() or (array[2:] <= 0).any():
        raise ValueError("bbox는 유한한 xywh 숫자 4개와 양수 너비·높이여야 합니다. 라벨표를 수정하세요.")
    return array.tolist()


def prepare_data(image_dir, annotation_dir, split_path, fixes_path):
    """원본은 보존하고 고정 분할과 add/modify 수정표를 적용한다.

    Args:
        image_dir (str | Path): 학습 이미지 폴더.
        annotation_dir (str | Path): COCO JSON 폴더.
        split_path (str | Path): file_name, combo, split 열의 고정 분할.
        fixes_path (str | Path): file_name, category_id, action, bbox 열의 수정표.

    Returns:
        dict: 이미지 목록·클래스 매핑·객체 라벨·데이터 식별자.
    """
    image_dir, annotation_dir = Path(image_dir), Path(annotation_dir)
    split_path, fixes_path = Path(split_path), Path(fixes_path)
    for path in (image_dir, annotation_dir, split_path, fixes_path):
        if not path.exists():
            raise FileNotFoundError(f"데이터 경로가 없습니다: {path}. 노트북 1번의 경로를 지정하세요.")
    json_paths = sorted(annotation_dir.rglob("*.json"))
    if not json_paths:
        raise ValueError("annotation JSON이 없습니다. train_annotations 경로를 확인하세요.")
    images, annotations, names, source_hashes = {}, [], {}, []
    for path in json_paths:
        doc = json.loads(path.read_text(encoding="utf-8-sig"))
        local_images = {int(item["id"]): item for item in doc["images"]}
        for item in local_images.values():
            name = item["file_name"]
            if Path(name).name != name:
                raise ValueError("이미지 파일명에 경로가 포함됐습니다. 이미지 manifest를 확인하세요.")
            record = {"file_name": name, "image_id": int(item["id"]),
                      "width": int(item["width"]), "height": int(item["height"])}
            if name in images and images[name] != record:
                raise ValueError(f"같은 이미지의 ID·크기가 다릅니다: {name}. 원본 연결을 확인하세요.")
            images[name] = record
        for category in doc.get("categories", []):
            names[int(category["id"])] = str(category.get("name") or category["id"])
        for ann in doc.get("annotations", []):
            if int(ann["image_id"]) not in local_images:
                raise ValueError(f"JSON 내부 image_id가 연결되지 않습니다: {path}. 라벨을 확인하세요.")
            annotations.append({"file_name": local_images[int(ann["image_id"])]["file_name"],
                                "category_id": int(ann["category_id"]), "bbox": _bbox(ann["bbox"]),
                                "iscrowd": int(ann.get("iscrowd", 0))})
        source_hashes.append((str(path.relative_to(annotation_dir)), file_sha256(path)))
    if not annotations:
        raise ValueError("객체 라벨이 없습니다. annotation 내용을 확인하세요.")
    raw_count = len(annotations)
    fixes = pd.read_csv(fixes_path, keep_default_na=False)
    if "bbox" not in fixes.columns and "bbox_after" in fixes.columns:
        fixes = fixes.rename(columns={"bbox_after": "bbox"})
    required = {"file_name", "category_id", "action", "bbox"}
    if not required.issubset(fixes.columns):
        raise ValueError(f"수정표에 {sorted(required)} 열이 필요합니다. label_fixes.csv를 확인하세요.")
    changed = Counter()
    for row in fixes.to_dict("records"):
        name, category = str(row["file_name"]).strip(), int(row["category_id"])
        if name not in images:
            raise ValueError(f"수정할 이미지가 원본에 없습니다: {name}. 수정표를 확인하세요.")
        bbox, action = _bbox(row["bbox"]), str(row["action"]).strip().lower()
        # 팀 공유표의 bbox·area도 기존 객체 수정으로 정규화한다.
        action = {"bbox": "modify", "area": "modify"}.get(action, action)
        matches = [a for a in annotations if a["file_name"] == name and a["category_id"] == category]
        if action == "add":
            if any(np.allclose(a["bbox"], bbox, atol=1e-6, rtol=0) for a in matches):
                raise ValueError(f"추가 라벨이 중복됩니다: {name}, {category}. 수정표를 확인하세요.")
            annotations.append({"file_name": name, "category_id": category, "bbox": bbox, "iscrowd": 0})
        elif action == "modify":
            if len(matches) != 1:
                raise ValueError(f"수정 대상이 {len(matches)}개입니다: {name}, {category}. 라벨을 구분하세요.")
            matches[0]["bbox"] = bbox
        else:
            raise ValueError(f"지원하지 않는 수정 action: {action}. add·modify·bbox·area로 지정하세요.")
        changed[action] += 1
    frame = pd.DataFrame(sorted(images.values(), key=lambda r: r["file_name"]))
    if frame["image_id"].duplicated().any():
        raise ValueError("서로 다른 이미지에 같은 image_id가 있습니다. 원본 연결을 확인하세요.")
    split = pd.read_csv(split_path, dtype=str, keep_default_na=False)
    if not {"file_name", "combo", "split"}.issubset(split.columns):
        raise ValueError("분할표에 file_name·combo·split 열이 필요합니다. 고정 분할표를 확인하세요.")
    if split["file_name"].duplicated().any() or set(split["file_name"]) != set(frame["file_name"]):
        raise ValueError("분할표의 이미지 목록이 원본과 다릅니다. 누락·중복을 확인하세요.")
    if set(split["split"]) != {"train", "val"}:
        raise ValueError("분할에는 train·val이 모두 있어야 합니다. 고정 분할표를 확인하세요.")
    if not split["combo"].eq(split["file_name"].str.split("_").str[0]).all():
        raise ValueError("분할표의 combo와 파일명 조합이 다릅니다. 고정 분할표를 확인하세요.")
    if split.groupby("combo")["split"].nunique().max() > 1:
        raise ValueError("같은 조합이 train·val에 섞였습니다. 팀의 고정 분할표를 확인하세요.")
    frame = frame.merge(split, on="file_name", validate="one_to_one")
    image_hashes = []
    for row in frame.to_dict("records"):
        path = image_dir / row["file_name"]
        with Image.open(path) as image:
            if image.size != (row["width"], row["height"]):
                raise ValueError(f"이미지와 라벨의 크기가 다릅니다: {path}. 좌표 기준을 확인하세요.")
        image_hashes.append((row["file_name"], file_sha256(path)))
    hash_splits = defaultdict(set)
    for name, digest in image_hashes:
        hash_splits[digest].add(split.set_index("file_name").loc[name, "split"])
    if any(len(parts) > 1 for parts in hash_splits.values()):
        raise ValueError("동일한 이미지 내용이 train·val에 중복됩니다. 고정 분할을 검토하세요.")
    categories = sorted({a["category_id"] for a in annotations})
    mapping = {category: index + 1 for index, category in enumerate(categories)}
    objects = defaultdict(list)
    for ann in annotations:
        info = images[ann["file_name"]]
        x, y, w, h = ann["bbox"]
        if x < 0 or y < 0 or x + w > info["width"] or y + h > info["height"]:
            raise ValueError(f"경계 밖 bbox: {ann['file_name']}, {ann['category_id']}. 수정표를 확인하세요.")
        if ann["iscrowd"] != 0:
            raise ValueError("이 실험은 개별 알약 iscrowd=0을 전제로 합니다. 평가 정책을 확인하세요.")
        xyxy = box_convert(torch.tensor([ann["bbox"]], dtype=torch.float64), "xywh", "xyxy")[0].tolist()
        objects[ann["file_name"]].append({"boxes": xyxy, "category_id": ann["category_id"],
                                          "label": mapping[ann["category_id"]]})
    class_map = pd.DataFrame({"category_id": categories,
                              "model_label": [mapping[c] for c in categories],
                              "name": [names.get(c, str(c)) for c in categories]})
    identity = stable_hash({"images": image_hashes, "annotations": source_hashes,
                            "split": file_sha256(split_path), "fixes": file_sha256(fixes_path),
                            "mapping": mapping})
    return {"frame": frame, "objects": dict(objects), "class_map": class_map,
            "category_to_label": mapping, "label_to_category": {v: k for k, v in mapping.items()},
            "image_dir": image_dir, "identity": identity, "image_hashes": dict(image_hashes),
            "raw_boxes": raw_count, "fixed_boxes": len(annotations), "fixes": dict(changed)}


def data_summary(bundle):
    """분할별 이미지 수와 클래스별 정답 객체 수를 집계한다.

    Args:
        bundle (dict): prepare_data 결과.

    Returns:
        tuple: 분할 요약표, 클래스별 객체 수.
    """
    rows = []
    for row in bundle["frame"].to_dict("records"):
        for obj in bundle["objects"].get(row["file_name"], []):
            rows.append({"split": row["split"], "category_id": obj["category_id"]})
    counts = pd.crosstab(pd.Series([r["category_id"] for r in rows], name="category_id"),
                         pd.Series([r["split"] for r in rows], name="split"))
    counts = counts.reindex(index=bundle["class_map"]["category_id"], columns=["train", "val"], fill_value=0)
    summary = pd.DataFrame([{"split": part, "images": int(bundle["frame"]["split"].eq(part).sum()),
                             "boxes": int(counts[part].sum()), "classes": int((counts[part] > 0).sum())}
                            for part in ("train", "val")])
    return summary, counts.reset_index()


def cache_images(bundle, cache_root):
    """내용 해시가 일치하는 로컬 이미지 캐시를 준비한다.

    Args:
        bundle (dict): 데이터 묶음.
        cache_root (str | Path): 복사본 저장 루트.

    Returns:
        dict: image_dir만 캐시 경로로 바꾼 묶음.
    """
    destination = Path(cache_root) / bundle["identity"][:12]
    destination.mkdir(parents=True, exist_ok=True)
    for name, digest in bundle["image_hashes"].items():
        target = destination / name
        if not target.exists() or file_sha256(target) != digest:
            with tempfile.NamedTemporaryFile(dir=destination, delete=False) as stream:
                temporary = Path(stream.name)
            shutil.copyfile(bundle["image_dir"] / name, temporary)
            if file_sha256(temporary) != digest:
                raise IOError("이미지 복사본이 원본과 다릅니다. 캐시를 다시 준비하세요.")
            os.replace(temporary, target)
    return {**bundle, "image_dir": destination}


class PillDataset(Dataset):
    """원본 픽셀 이미지와 torchvision용 정답을 반환한다."""

    def __init__(self, frame, image_dir, objects=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.objects = objects

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row["file_name"]) as image:
            tensor = to_tensor(image.convert("RGB"))
        objects = [] if self.objects is None else self.objects.get(row["file_name"], [])
        boxes = torch.tensor([o["boxes"] for o in objects], dtype=torch.float32).reshape(-1, 4)
        return tensor, {"boxes": boxes, "labels": torch.tensor([o["label"] for o in objects], dtype=torch.int64),
                        "image_id": torch.tensor(int(row["image_id"])),
                        "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
                        "iscrowd": torch.zeros(len(objects), dtype=torch.int64)}


def collate_detection(batch):
    """객체 수가 다른 이미지와 정답을 목록으로 묶는다.

    Args:
        batch (list): 이미지·정답 쌍.

    Returns:
        tuple: 이미지 목록, 정답 목록.
    """
    return tuple(zip(*batch))


def make_dataset(bundle, split):
    """지정된 고정 분할의 Dataset을 만든다.

    Args:
        bundle (dict): 데이터 묶음.
        split (str): train 또는 val.

    Returns:
        PillDataset: 선택 분할.
    """
    return PillDataset(bundle["frame"].loc[bundle["frame"]["split"].eq(split)],
                       bundle["image_dir"], bundle["objects"])


def cache_dataset(dataset, cache_root):
    """Test를 포함한 Dataset의 이미지를 검증된 로컬 캐시로 연결한다.

    Args:
        dataset (PillDataset): 복사할 이미지 목록.
        cache_root (str | Path): 캐시 루트.

    Returns:
        PillDataset: 같은 순서·정답을 유지하는 로컬 Dataset.
    """
    hashes = {name: file_sha256(dataset.image_dir / name) for name in dataset.frame["file_name"]}
    source = {"image_dir": dataset.image_dir, "image_hashes": hashes,
              "identity": stable_hash({"files": hashes, "frame": dataset.frame.to_dict("records")})}
    cached = cache_images(source, cache_root)
    return PillDataset(dataset.frame, cached["image_dir"], dataset.objects)


def test_dataset(image_dir):
    """숫자 파일명을 대회 image_id로 보존한 Test Dataset을 만든다.

    Args:
        image_dir (str | Path): Test 이미지 폴더.

    Returns:
        PillDataset: 정답이 없는 이미지 목록.
    """
    image_dir = Path(image_dir)
    rows = []
    for path in sorted(image_dir.iterdir()):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
            continue
        if not path.stem.isdigit():
            raise ValueError(f"Test 파일명이 숫자가 아닙니다: {path.name}. 대회 파일을 확인하세요.")
        with Image.open(path) as image:
            width, height = image.size
        rows.append({"file_name": path.name, "image_id": int(path.stem), "width": width, "height": height})
    frame = pd.DataFrame(rows)
    if frame.empty or frame["image_id"].duplicated().any():
        raise ValueError("Test 이미지가 없거나 image_id가 중복됩니다. 경로·파일명을 확인하세요.")
    return PillDataset(frame.sort_values("image_id"), image_dir)


'''

INLINE_ANALYSIS_BLOCKS['analysis'] = r'''
"""모델에 독립적인 원본 좌표 평가·오류 분석·제출 변환."""

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou, box_convert




IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
SUBMISSION_COLUMNS = ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y",
                      "bbox_w", "bbox_h", "score"]


def evaluate_records(records, class_map=None, detailed=True):
    """공통 예측의 bbox mAP와 선택적 클래스·IoU별 AP를 계산한다.

    Args:
        records (list): prediction과 target을 포함한 이미지별 결과. labels는 원래 category ID.
        class_map (DataFrame | None): 전체 클래스 목록.
        detailed (bool): 상세 결과 계산 여부.

    Returns:
        dict: 요약, 클래스별 AP, IoU별 AP, PR 곡선 원자료.
    """
    if not records or any("target" not in row for row in records):
        raise ValueError("정답이 있는 validation 결과가 필요합니다. 4번 검증 추론을 실행하세요.")
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=IOU_THRESHOLDS,
                                  max_detection_thresholds=[1, 10, 100], class_metrics=detailed,
                                  extended_summary=detailed)
    metric.update([row["prediction"] for row in records], [row["target"] for row in records])
    result = metric.compute()
    summary = {"map_75_95": float(result["map"]), "map_75": float(result["map_75"]),
               "mar_100": float(result["mar_100"]), "images": len(records),
               "ground_truths": sum(len(r["target"]["boxes"]) for r in records)}
    if not detailed:
        return {"summary": summary}
    counts = Counter(int(c) for row in records for c in row["target"]["labels"])
    precision = result["precision"][:, :, :, 0, -1].numpy()
    classes = result["classes"].reshape(-1).tolist()

    def valid_mean(values):
        valid = values[values >= 0]
        return float(valid.mean()) if valid.size else None

    ap = pd.DataFrame([{"category_id": int(category), "ap_75_95": valid_mean(precision[:, :, index]),
                        "val_objects": counts[int(category)]} for index, category in enumerate(classes)])
    if class_map is not None:
        ap = class_map[["category_id", "name"]].merge(ap, how="left", on="category_id")
        ap["val_objects"] = ap["val_objects"].fillna(0).astype(int)
    ap.loc[ap["val_objects"].eq(0), "ap_75_95"] = np.nan
    iou_ap = pd.DataFrame({"iou": IOU_THRESHOLDS,
                           "ap": [valid_mean(precision[index]) for index in range(len(IOU_THRESHOLDS))]})
    return {"summary": summary, "classes": ap, "ious": iou_ap,
            "precision": precision, "category_ids": classes}


def detection_errors(records, categories, iou_threshold=0.75, score_threshold=0.30):
    """클래스와 독립적인 IoU 일대일 대응으로 혼동과 미검출을 집계한다.

    Args:
        records (list): validation 공통 결과.
        categories (list): 원래 category ID 목록.
        iou_threshold (float): 대응 최소 IoU.
        score_threshold (float): 진단용 최소 score.

    Returns:
        tuple: 행=정답·열=예측 혼동행렬, 오류 상세 표.
    """
    if not 0 < iou_threshold <= 1 or not 0 <= score_threshold <= 1:
        raise ValueError("진단 IoU·score 범위가 잘못됐습니다. 4번 설정을 확인하세요.")
    categories = [int(c) for c in categories]
    positions = {category: index for index, category in enumerate(categories)}
    background = len(categories)
    matrix = np.zeros((background + 1, background + 1), dtype=int)
    events = []
    for record in records:
        target, pred = record["target"], record["prediction"]
        kept = torch.where(pred["scores"] >= score_threshold)[0]
        boxes, labels, scores = pred["boxes"][kept], pred["labels"][kept], pred["scores"][kept]
        overlaps = box_iou(target["boxes"], boxes).numpy()
        candidates = np.argwhere(overlaps >= iou_threshold)
        candidates = sorted(candidates.tolist(), key=lambda pair: (-overlaps[tuple(pair)], pair))
        used_gt, used_pred = set(), set()

        def append(kind, gt=None, prediction=None, overlap=None):
            events.append({"image_id": int(record["image_id"]), "file_name": record["file_name"],
                           "kind": kind, "gt_index": gt,
                           "pred_index": None if prediction is None else int(kept[prediction]),
                           "true_category": None if gt is None else int(target["labels"][gt]),
                           "pred_category": None if prediction is None else int(labels[prediction]),
                           "score": None if prediction is None else float(scores[prediction]),
                           "iou": overlap})

        for gt, prediction in candidates:
            if gt in used_gt or prediction in used_pred:
                continue
            used_gt.add(gt)
            used_pred.add(prediction)
            truth, estimate = int(target["labels"][gt]), int(labels[prediction])
            matrix[positions[truth], positions[estimate]] += 1
            append("correct" if truth == estimate else "class_confusion", gt, prediction,
                   float(overlaps[gt, prediction]))
        for gt in range(len(target["boxes"])):
            if gt not in used_gt:
                matrix[positions[int(target["labels"][gt])], background] += 1
                possible = [p for p in range(len(boxes)) if int(labels[p]) == int(target["labels"][gt])
                            and 0 < overlaps[gt, p] < iou_threshold]
                best = max(possible, key=lambda p: overlaps[gt, p]) if possible else None
                append("localization_or_missed" if best is not None else "missed", gt, best,
                       None if best is None else float(overlaps[gt, best]))
        for prediction in range(len(boxes)):
            if prediction not in used_pred:
                matrix[background, positions[int(labels[prediction])]] += 1
                duplicate = any(int(target["labels"][gt]) == int(labels[prediction])
                                and overlaps[gt, prediction] >= iou_threshold for gt in used_gt)
                append("duplicate" if duplicate else "extra", prediction=prediction)
    labels = [str(c) for c in categories] + ["unmatched"]
    columns = ["image_id", "file_name", "kind", "gt_index", "pred_index", "true_category",
               "pred_category", "score", "iou"]
    return pd.DataFrame(matrix, index=labels, columns=labels), pd.DataFrame(events, columns=columns)


def plot_ground_truth(dataset, reverse, count=3):
    """입력 데이터의 정답 bbox를 표시한다.

    Args:
        dataset (PillDataset): 정답이 있는 이미지.
        reverse (dict): 모델 번호에서 원래 클래스 ID로 가는 매핑.
        count (int): 표시할 이미지 수.

    Returns:
        Figure: 정답 박스 이미지.
    """
    fig, axes = plt.subplots(1, min(count, len(dataset)), figsize=(6 * min(count, len(dataset)), 6), squeeze=False)
    for index, axis in enumerate(axes[0]):
        _, target = dataset[index]
        image = draw_boxes(dataset.image_dir / dataset.frame.iloc[index]["file_name"],
                           [xyxy_to_xywh(box.tolist()) for box in target["boxes"]],
                           [str(reverse[int(c)]) for c in target["labels"]], color="lime")
        axis.imshow(image)
        axis.set(title=dataset.frame.iloc[index]["file_name"][:30])
        axis.axis("off")
    fig.tight_layout()
    return fig


def plot_learning_curves(run_dirs):
    """완료 여부와 관계없이 저장된 학습 곡선을 표시한다.

    Args:
        run_dirs (list): metrics.csv를 포함한 실행 폴더.

    Returns:
        Figure: loss·mAP 곡선.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for directory in run_dirs:
        path = Path(directory) / "metrics.csv"
        if not path.exists():
            continue
        history = pd.read_csv(path)
        if history.empty:
            continue
        name = Path(directory).name
        axes[0].plot(history["epoch"], history["train_loss"], label=name)
        axes[1].plot(history["epoch"], history["val_map_75_95"], label=name)
    for axis, title in zip(axes, ["Train loss", "Validation mAP@[.75:.95]"]):
        axis.set(title=title, xlabel="Epoch")
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend(fontsize=7)
    fig.tight_layout()
    return fig


def plot_candidate_scores(results):
    """설정별 seed 점수와 평균·표본 표준편차를 함께 표시한다.

    Args:
        results (DataFrame): 완료된 실행 결과.

    Returns:
        Figure: seed별 점수와 설정별 평균. 단일 seed의 표준편차는 추정하지 않음.
    """
    summary = results.groupby("candidate_id")["best_val_map_75_95"].agg(["mean", "std", "count"])
    summary = summary.sort_values("mean", ascending=False)
    positions = {key: index for index, key in enumerate(summary.index)}
    x = np.arange(len(summary))
    fig, axis = plt.subplots(figsize=(max(8, len(summary)*1.2), 5))
    axis.bar(x, summary["mean"], color="lightsteelblue", label="Mean")
    estimated = summary["count"].ge(2).to_numpy()
    axis.errorbar(x[estimated], summary.loc[estimated, "mean"], yerr=summary.loc[estimated, "std"],
                  fmt="none", ecolor="black", capsize=4, label="Sample SD (n>=2)")
    for seed, group in results.groupby("seed"):
        axis.scatter([positions[key] for key in group["candidate_id"]], group["best_val_map_75_95"],
                     label=f"seed {int(seed)}", zorder=3)
    axis.set(xticks=x, xticklabels=[f"{key}\nn={int(summary.loc[key, 'count'])}" for key in summary.index],
             ylabel="Validation mAP@[.75:.95]", ylim=(0, 1.05),
             title="Candidate and seed comparison (n=1: SD unavailable)")
    axis.tick_params(axis="x", rotation=30, labelsize=8)
    axis.legend(fontsize=8)
    axis.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    return fig


def plot_ap(report):
    """낮은 클래스 AP·정답 수와 IoU별 AP를 함께 표시한다.

    Args:
        report (dict): 상세 evaluate_records 결과.

    Returns:
        Figure: 클래스·IoU별 그래프.
    """
    low = report["classes"].dropna(subset=["ap_75_95"]).nsmallest(15, "ap_75_95")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh([f"{int(row.category_id)} (n={int(row.val_objects)})" for row in low.itertuples()], low["ap_75_95"])
    axes[0].invert_yaxis()
    axes[0].set(title="Lowest class AP (validation support)", xlabel="AP@[.75:.95]", xlim=(0, 1))
    axes[1].plot(report["ious"]["iou"], report["ious"]["ap"], marker="o")
    axes[1].set(title="AP by IoU", xlabel="IoU", ylabel="AP", ylim=(0, 1))
    axes[1].grid(alpha=0.25)
    fig.tight_layout()
    return fig


def plot_confusion(matrix, normalize=True):
    """정답 행 기준 정규화 또는 원시 건수 혼동행렬을 표시한다.

    Args:
        matrix (DataFrame): 행=정답·열=예측. unmatched 행은 추가 예측, 열은 미검출.
        normalize (bool): 정답 행별 비율 표시 여부.

    Returns:
        Figure: 진단 혼동행렬.
    """
    values = matrix.to_numpy(dtype=float)
    if normalize:
        values = np.divide(values, values.sum(axis=1, keepdims=True),
                           out=np.zeros_like(values), where=values.sum(axis=1, keepdims=True) > 0)
    size = max(9, min(20, len(matrix) * 0.28))
    fig, axis = plt.subplots(figsize=(size, size))
    heatmap = axis.imshow(values, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    axis.set(xticks=range(len(matrix)), yticks=range(len(matrix)),
             xticklabels=matrix.columns, yticklabels=matrix.index,
             xlabel="Predicted (unmatched = missed)", ylabel="True (unmatched = extra)",
             title="Detection confusion: row-normalized" if normalize else "Detection confusion: counts")
    axis.tick_params(axis="x", rotation=90, labelsize=7)
    axis.tick_params(axis="y", labelsize=7)
    fig.colorbar(heatmap, ax=axis, fraction=0.04)
    fig.tight_layout()
    return fig


def preview_records(records, events=None, count=6, score_threshold=0.30):
    """정답·예측과 오류 종류를 표시한다. Test에는 정답을 그리지 않는다.

    Args:
        records (list): 공통 예측 결과.
        events (DataFrame | None): validation 오류 표.
        count (int): 최대 이미지 수.
        score_threshold (float): 표시용 점수 하한.

    Returns:
        Figure: 초록 정답·빨강 예측 이미지.
    """
    if events is not None:
        errors = events.loc[events["kind"].ne("correct")].copy()
        errors = errors.sort_values("score", ascending=False, na_position="last")
        ids = list(dict.fromkeys(errors["image_id"].tolist()))[:count]
        chosen = [r for identifier in ids for r in records if r["image_id"] == identifier]
    else:
        chosen = records[:count]
    if not chosen:
        fig, axis = plt.subplots(figsize=(8, 2))
        axis.text(0.5, 0.5, "No error images at the chosen diagnostic thresholds", ha="center")
        axis.axis("off")
        return fig
    fig, axes = plt.subplots(len(chosen), 1, figsize=(9, 6 * len(chosen)), squeeze=False)
    for record, axis in zip(chosen, axes[:, 0]):
        with Image.open(record["image_path"]) as source:
            image = source.convert("RGB")
        if "target" in record:
            target = record["target"]
            image = draw_boxes(image, [xyxy_to_xywh(b.tolist()) for b in target["boxes"]],
                               [f"GT:{int(c)}" for c in target["labels"]], color="lime")
        pred = record["prediction"]
        indices = torch.where(pred["scores"] >= score_threshold)[0]
        indices = indices[torch.argsort(pred["scores"][indices], descending=True)][:20]
        captions = []
        for index in indices:
            caption = f"P:{int(pred['labels'][index])} {float(pred['scores'][index]):.2f}"
            if "target" in record and len(record["target"]["boxes"]):
                overlap = float(box_iou(pred["boxes"][index].reshape(1, 4), record["target"]["boxes"]).max())
                caption += f" I:{overlap:.2f}"
            captions.append(caption)
        image = draw_boxes(image, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in indices],
                           captions, color="red")
        image.thumbnail((1280, 1280))
        kinds = "prediction only; no ground truth"
        if events is not None:
            kinds = ", ".join(sorted(set(events.loc[events["image_id"].eq(record["image_id"]), "kind"])))
        axis.imshow(image)
        axis.set_title(f"ID {record['image_id']} | {kinds}\ngreen=GT, red=prediction; score>={score_threshold}", fontsize=9)
        axis.axis("off")
    fig.tight_layout()
    return fig


def submission_frame(records, allowed_categories):
    """공통 원본 좌표 예측을 대회 CSV 표로 변환한다.

    Args:
        records (list): labels가 원래 category ID인 공통 예측.
        allowed_categories (list): 허용 클래스 목록.

    Returns:
        DataFrame: 대회 컬럼 순서의 표. 빈 예측도 빈 표로 표현한다.
    """
    allowed_categories = set(int(c) for c in allowed_categories)
    if len({int(r["image_id"]) for r in records}) != len(records):
        raise ValueError("예측 이미지 ID가 중복됩니다. 전체 추론 목록을 확인하세요.")
    rows = []
    for record in records:
        pred = record["prediction"]
        if not (len(pred["boxes"]) == len(pred["labels"]) == len(pred["scores"])):
            raise ValueError("예측 boxes·labels·scores 수가 다릅니다. 공통 변환을 확인하세요.")
        for box, category, score in zip(pred["boxes"], pred["labels"], pred["scores"]):
            x, y, w, h = xyxy_to_xywh(box.tolist())
            if int(category) not in allowed_categories:
                raise ValueError("알 수 없는 category ID입니다. 모델의 클래스 매핑을 확인하세요.")
            if not np.isfinite([x, y, w, h, float(score)]).all() or not 0 <= float(score) <= 1:
                raise ValueError("유한하지 않은 좌표·score가 있습니다. 모델 출력을 확인하세요.")
            if x < -1e-4 or y < -1e-4 or w <= 0 or h <= 0 or x+w > record["width"]+1e-4 or y+h > record["height"]+1e-4:
                raise ValueError("원본 경계 밖 또는 면적 0 예측입니다. 좌표 복원을 확인하세요.")
            rows.append({"image_id": int(record["image_id"]), "category_id": int(category),
                         "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h, "score": float(score)})
    frame = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS[1:])
    frame = frame.sort_values(["image_id", "score"], ascending=[True, False], kind="stable").reset_index(drop=True)
    frame.insert(0, "annotation_id", np.arange(1, len(frame)+1, dtype=np.int64))
    for column in ("annotation_id", "image_id", "category_id"):
        frame[column] = frame[column].astype("int64")
    return frame[SUBMISSION_COLUMNS]


def restore_submission(frame, records):
    """CSV를 다시 읽은 뒤 이미지별 공통 예측으로 복원한다.

    Args:
        frame (DataFrame): 제출 컬럼의 표.
        records (list): 전체 이미지·정답 목록.

    Returns:
        list: 빈 이미지도 보존한 복원 결과.
    """
    if list(frame.columns) != SUBMISSION_COLUMNS or frame.isna().any().any():
        raise ValueError("제출 컬럼·결측값이 잘못됐습니다. CSV 생성 코드를 확인하세요.")
    try:
        frame = frame.apply(pd.to_numeric, errors="raise")
    except (TypeError, ValueError) as error:
        raise ValueError("CSV에 숫자가 아닌 값이 있습니다. 제출 파일을 다시 생성하세요.") from error
    if not np.isfinite(frame.to_numpy(dtype=float)).all():
        raise ValueError("CSV에 유한하지 않은 값이 있습니다. 제출 파일을 다시 생성하세요.")
    for column in ("annotation_id", "image_id", "category_id"):
        if not np.equal(frame[column], np.floor(frame[column])).all():
            raise ValueError("CSV ID는 정수여야 합니다. 클래스·이미지 매핑을 확인하세요.")
    if frame["annotation_id"].duplicated().any() or not set(frame["image_id"]).issubset({r["image_id"] for r in records}):
        raise ValueError("제출 ID가 중복되거나 이미지 목록과 다릅니다. CSV를 확인하세요.")
    restored = []
    for record in records:
        rows = frame.loc[frame["image_id"].eq(record["image_id"])]
        boxes = torch.tensor(rows[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]].to_numpy(dtype=np.float32), dtype=torch.float32).reshape(-1, 4)
        restored.append({**record, "prediction": {
            "boxes": box_convert(boxes, "xywh", "xyxy"),
            "labels": torch.tensor(rows["category_id"].to_numpy(dtype=np.int64), dtype=torch.int64),
            "scores": torch.tensor(rows["score"].to_numpy(dtype=np.float32), dtype=torch.float32)}})
    return restored


def verify_roundtrip(records, categories, csv_path):
    """실제 CSV 저장·읽기와 전후 mAP·좌표·ID 일치를 검사한다.

    Args:
        records (list): 정답을 포함한 validation 예측.
        categories (list): 원래 클래스 ID 목록.
        csv_path (str | Path): 검증용 CSV 저장 위치.

    Returns:
        dict: 변환 전후 mAP와 통과 결과.
    """
    path = Path(csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = submission_frame(records, categories)
    frame.to_csv(path, index=False)
    loaded = pd.read_csv(path)
    restored = restore_submission(loaded, records)
    for before, after in zip(records, restored):
        left, right = before["prediction"], after["prediction"]
        order = torch.argsort(left["scores"], descending=True, stable=True)
        if not torch.equal(left["labels"][order], right["labels"]) or not torch.allclose(left["boxes"][order], right["boxes"], atol=1e-3, rtol=0) or not torch.allclose(left["scores"][order], right["scores"], atol=1e-7, rtol=0):
            raise ValueError("CSV 복원 전후 예측이 바뀌었습니다. ID·좌표 변환을 확인하세요.")
    before = evaluate_records(records, detailed=False)["summary"]["map_75_95"]
    after = evaluate_records(restored, detailed=False)["summary"]["map_75_95"]
    if abs(before-after) > 1e-6:
        raise ValueError("CSV 변환 전후 mAP가 다릅니다. 평가 입력과 좌표 변환을 확인하세요.")
    report = {"before": before, "after": after, "passed": True, "images": len(records), "rows": len(frame)}
    write_json(path.with_suffix(".json"), report)
    return report


'''

INLINE_ANALYSIS_BLOCKS['reports_and_loss'] = r'''
"""Kaggle에서 저장된 seed 0 결과를 분석한다. 학습·가중치 갱신은 수행하지 않는다."""

import contextlib
import gc
import importlib.metadata
import math
import random
import subprocess
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d, nms


def save_table(frame, path, index=False):
    """한글을 포함하는 분석 표를 Excel에서도 읽을 수 있게 저장한다.

    Args:
        frame (DataFrame): 저장할 표.
        path (str | Path): 출력 CSV 경로.
        index (bool): 행 이름 저장 여부.
    """
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=index, encoding="utf-8-sig")


def save_figure(figure, path):
    """그래프를 저장한 뒤 메모리에서 해제한다.

    Args:
        figure (Figure): 완성한 그래프.
        path (str | Path): 출력 이미지 경로.
    """
    figure.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(figure)


def flatten_settings(value, prefix=""):
    """중첩된 설정의 모든 항목을 경로와 값으로 펼친다.

    Args:
        value (object): 설정 또는 환경 정보.
        prefix (str): 중첩 키의 경로.

    Returns:
        list: setting, value를 담은 행 목록.
    """
    if isinstance(value, dict):
        rows = []
        for key in sorted(value):
            rows.extend(flatten_settings(value[key], f"{prefix}.{key}" if prefix else str(key)))
        return rows
    return [{"setting": prefix, "value": json.dumps(value, ensure_ascii=False, allow_nan=False)}]


def measurement_environment():
    """재측정 환경을 원래 학습 환경과 구분해서 기록한다.

    Returns:
        dict: Python, 패키지, CUDA, GPU 정보.
    """
    names = ["torch", "torchvision", "torchmetrics", "numpy", "pandas", "pycocotools", "pillow", "matplotlib"]
    return {"python": sys.version.split()[0],
            "packages": {name: importlib.metadata.version(name) for name in names},
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}


def load_run_context(job):
    """동일 실행의 설정·이력·완료 결과를 읽고 일치 여부를 검사한다.

    Args:
        job (dict): 입력 실행 경로와 분석 설정.

    Returns:
        dict: 검증한 입력 자료와 출력 경로.
    """
    run = Path(job["run_dir"]).resolve()
    for name in ["config.json", "metrics.csv", "result.json", "class_map.csv"]:
        if not (run / name).is_file():
            raise FileNotFoundError(f"{name}이 없습니다. Drive의 같은 실행 폴더 전체를 업로드하세요: {run}")
    manifest = json.loads((run / "config.json").read_text())
    spec, recipe = manifest["spec"], manifest["recipe"]
    if spec["seed"] != 0 or spec["mode"] != "ft" or not math.isclose(spec["lr"], 3e-4, rel_tol=0, abs_tol=1e-12):
        raise ValueError("분석 대상이 FT / 3e-4 / seed 0이 아닙니다. RUN_DIR을 확인하세요.")
    if stable_hash({"recipe": recipe, "spec": spec}) != manifest["signature"]:
        raise ValueError("config.json의 식별자가 일치하지 않습니다. 원본 실행 폴더를 다시 업로드하세요.")
    result = json.loads((run / "result.json").read_text())
    for key in ["spec_id", "candidate_id", "mode", "seed", "lr"]:
        if result[key] != spec[key]:
            raise ValueError(f"result.json과 config.json의 {key}가 다릅니다. 같은 실행의 파일을 준비하세요.")
    if result["run_name"] != manifest["run_name"]:
        raise ValueError("실행 이름이 일치하지 않습니다. 다른 실행 결과를 섞지 마세요.")
    history = pd.read_csv(run / "metrics.csv")
    needed = ["epoch", "train_loss", "val_map_75_95"]
    if not set(needed).issubset(history) or history.empty:
        raise ValueError("metrics.csv에 epoch·train_loss·val_map_75_95 이력이 필요합니다.")
    if not np.isfinite(history[needed].to_numpy(dtype=float)).all():
        raise ValueError("학습 이력에 유한하지 않은 수치가 있습니다. 원본 CSV를 확인하세요.")
    if (history["epoch"].duplicated().any() or not history["epoch"].is_monotonic_increasing
            or not history["epoch"].ge(1).all() or not (history["epoch"] % 1 == 0).all()):
        raise ValueError("epoch 순서가 잘못됐거나 중복됐습니다. 원본 CSV를 확인하세요.")
    if not history["val_map_75_95"].between(0, 1).all() or not history["train_loss"].ge(0).all():
        raise ValueError("mAP 또는 loss 범위가 잘못됐습니다. 원본 지표를 확인하세요.")
    best_row = history.loc[history["val_map_75_95"].idxmax()]
    if (int(best_row["epoch"]) != result["best_epoch"]
            or int(history.iloc[-1]["epoch"]) != result["last_epoch"]
            or not math.isclose(float(best_row["val_map_75_95"]), result["best_val_map_75_95"], abs_tol=1e-6)):
        raise ValueError("최고·마지막 epoch 또는 최고 mAP가 result.json과 다릅니다. 같은 실행 파일인지 확인하세요.")
    classes = pd.read_csv(run / "class_map.csv")
    if not {"category_id", "model_label", "name"}.issubset(classes) or classes["category_id"].duplicated().any():
        raise ValueError("class_map.csv 형식이 잘못됐습니다. 원본 클래스 번호표를 준비하세요.")
    expected = {int(k): int(v) for k, v in recipe["class_mapping"].items()}
    found = dict(zip(classes["category_id"].astype(int), classes["model_label"].astype(int)))
    if found != expected or sorted(found.values()) != list(range(1, len(found) + 1)):
        raise ValueError("클래스 매핑이 학습 기록과 다릅니다. 원본 번호표를 사용하세요.")
    out = Path(job["output_dir"]).resolve()
    if out == run or run in out.parents:
        raise ValueError("분석 출력은 입력 실행 폴더 밖에 저장하세요. OUTPUT_ROOT를 바꾸세요.")
    out.mkdir(parents=True, exist_ok=True)
    return {"run": run, "out": out, "manifest": manifest, "config": recipe["config"],
            "result": result, "history": history, "classes": classes}


def verified_dataset(context, job):
    """원본 이미지·수정 라벨·고정 분할이 학습 때와 같은지 해시로 확인한다.

    Args:
        context (dict): 실행 자료.
        job (dict): 이미지·annotation·분할·수정표 경로.

    Returns:
        dict: 검증된 데이터 묶음.
    """
    paths = [job.get(k) for k in ["image_dir", "annotation_dir", "split_path", "fixes_path"]]
    if not all(paths):
        raise FileNotFoundError("손실 재측정에는 원본 이미지·annotation·split.csv·사용했던 수정표가 모두 필요합니다. 셀 1 경로를 지정하세요.")
    print("원본 데이터·분할·수정표 식별자 확인 중…", flush=True)
    bundle = prepare_data(*paths)
    manifest = context["manifest"]
    if bundle["identity"] != manifest["recipe"]["data_identity"]:
        raise ValueError("데이터 해시가 학습 기록과 다릅니다. 이미지 내용·JSON 폴더 구조·분할·수정표를 바꾸지 말고 원본을 업로드하세요.")
    expected = {int(k): int(v) for k, v in manifest["recipe"]["class_mapping"].items()}
    if bundle["category_to_label"] != expected:
        raise ValueError("모델 라벨 매핑이 학습 당시와 다릅니다. 원본 데이터·번호표를 확인하세요.")
    summary, counts = data_summary(bundle)
    save_table(summary, context["out"] / "dataset_summary.csv")
    save_table(counts, context["out"] / "dataset_class_support.csv")
    return bundle


def load_cached_records(context, bundle=None):
    """저장된 검증 예측을 읽고 좌표·클래스·가능한 경우 캐시 식별자도 검사한다.

    Args:
        context (dict): 실행 자료.
        bundle (dict | None): 원본 데이터 검증 결과.

    Returns:
        list: 원래 category ID 기준 정답과 예측.
    """
    path = context["run"] / "validation_predictions.pt"
    if not path.is_file():
        raise FileNotFoundError("validation_predictions.pt가 없습니다. 원본 실행 폴더의 검증 예측 파일을 업로드하세요.")
    # 사용자가 직접 생성한 체크포인트·예측 파일만 입력으로 받는다.
    payload = torch.load(path, map_location="cpu", weights_only=False)
    records = payload["records"]
    if not records or len({r["image_id"] for r in records}) != len(records):
        raise ValueError("검증 예측이 비었거나 image_id가 중복됐습니다. 원본 예측 파일을 준비하세요.")
    categories = set(context["classes"]["category_id"].astype(int))
    for row in records:
        for name in ["target", "prediction"]:
            item = row[name]
            boxes, labels = item["boxes"], item["labels"]
            if boxes.ndim != 2 or boxes.shape[1] != 4 or len(labels) != len(boxes):
                raise ValueError("예측·정답 텐서 형식이 잘못됐습니다. 원본 예측 파일을 준비하세요.")
            if not torch.isfinite(boxes).all() or not set(labels.tolist()).issubset(categories):
                raise ValueError("예측·정답의 좌표 또는 클래스가 잘못됐습니다. 번호표와 예측 파일을 확인하세요.")
            if len(boxes) and (not (boxes[:, 2:] > boxes[:, :2]).all() or (boxes < 0).any()
                               or (boxes[:, [0, 2]] > row["width"]).any()
                               or (boxes[:, [1, 3]] > row["height"]).any()):
                raise ValueError("원본 이미지 경계를 벗어나거나 뒤집힌 bbox가 있습니다. 예측 파일을 확인하세요.")
        pred = row["prediction"]
        if (len(pred["scores"]) != len(pred["boxes"]) or not torch.isfinite(pred["scores"]).all()
                or not ((pred["scores"] >= 0) & (pred["scores"] <= 1)).all()):
            raise ValueError("예측 score가 잘못됐습니다. 원본 예측 파일을 확인하세요.")
    provenance = {"original_run_signature": context["manifest"]["signature"],
                  "prediction_sha256": file_sha256(path), "images": len(records),
                  "verified_against_original_images_and_best_checkpoint": False}
    if bundle is not None:
        dataset = make_dataset(bundle, "val")
        frame = dataset.frame
        if [(r["file_name"], r["image_id"]) for r in records] != list(zip(frame["file_name"], frame["image_id"])):
            raise ValueError("검증 예측의 이미지 목록·순서가 고정 분할과 다릅니다. 같은 실행 파일을 사용하세요.")
        for index, record in enumerate(records):
            _, target = dataset[index]
            original_labels = torch.tensor([bundle["label_to_category"][int(v)] for v in target["labels"]], dtype=torch.int64)
            if (not torch.equal(record["target"]["boxes"], target["boxes"])
                    or not torch.equal(record["target"]["labels"], original_labels)):
                raise ValueError("저장된 검증 정답이 원본 라벨과 다릅니다. 데이터·예측 파일을 확인하세요.")
            record["image_path"] = str(dataset.image_dir / record["file_name"])
        image_identity = [(r.file_name, bundle["image_hashes"][r.file_name]) for r in frame.itertuples()]
        identity = stable_hash({"checkpoint": file_sha256(context["run"] / "best.pt"), "images": image_identity,
                                "frame": frame[["file_name", "image_id", "width", "height"]].to_dict("records"),
                                "with_targets": True, "signature": context["manifest"]["signature"]})
        if payload["identity"] != identity:
            raise ValueError("예측 캐시의 식별자가 best.pt·검증 이미지와 일치하지 않습니다. 같은 실행 파일을 준비하세요.")
        provenance["verified_against_original_images_and_best_checkpoint"] = True
    write_json(context["out"] / "prediction_provenance.json", provenance)
    return records


def diagnostic_tables(matrix, classes):
    """고정 IoU·score에서 클래스별 오류와 혼동 쌍을 집계한다.

    Args:
        matrix (DataFrame): 배경 행·열을 포함한 검출 혼동행렬.
        classes (DataFrame): 전체 클래스 번호표.

    Returns:
        tuple: 클래스 요약, 혼동 쌍 목록.
    """
    rows, pairs = [], []
    names = dict(zip(classes["category_id"].astype(int), classes["name"].astype(str)))
    categories = classes["category_id"].astype(int).tolist()
    for category in categories:
        key = str(category)
        tp = int(matrix.loc[key, key])
        gt, predictions = int(matrix.loc[key].sum()), int(matrix[key].sum())
        unmatched = int(matrix.loc[key, "unmatched"])
        confusion = gt - tp - unmatched
        precision = tp / predictions if predictions else np.nan
        recall = tp / gt if gt else np.nan
        f1 = 2 * tp / (gt + predictions) if gt + predictions else np.nan
        rows.append({"category_id": category, "name": names[category], "ground_truths": gt,
                     "predictions": predictions, "correct": tp, "wrong_class_matches": confusion,
                     "unmatched_ground_truths": unmatched, "total_false_negatives": gt - tp,
                     "total_false_positives": predictions - tp,
                     "diagnostic_precision": precision, "diagnostic_recall": recall,
                     "diagnostic_f1": f1, "diagnostic_error_rate": (gt - tp) / gt if gt else np.nan})
        for estimate in categories:
            if estimate == category:
                continue
            count = int(matrix.loc[key, str(estimate)])
            if count:
                pairs.append({"true_category": category, "true_name": names[category],
                              "pred_category": estimate, "pred_name": names[estimate], "count": count,
                              "fraction_of_true_objects": count / gt if gt else np.nan})
    summary = pd.DataFrame(rows).sort_values(["total_false_negatives", "ground_truths"], ascending=False)
    pair_columns = ["true_category", "true_name", "pred_category", "pred_name", "count", "fraction_of_true_objects"]
    return summary, pd.DataFrame(pairs, columns=pair_columns).sort_values("count", ascending=False)


def saved_result_reports(job):
    """기존 CSV·검증 예측으로 요청된 성능 표·진단 그래프를 생성한다.

    Args:
        job (dict): 입력·출력 경로와 진단 기준.
    """
    context = load_run_context(job)
    out, history = context["out"], context["history"]
    bundle = verified_dataset(context, job) if all(job.get(k) for k in ["image_dir", "annotation_dir", "split_path", "fixes_path"]) else None
    records = load_cached_records(context, bundle)
    settings = {"original_manifest": context["manifest"], "original_result": context["result"],
                "analysis_environment": measurement_environment(),
                "analysis_policy": {"iou_thresholds": IOU_THRESHOLDS,
                                    "max_detections": [1, 10, 100],
                                    "diagnostic_iou": job["diagnostic_iou"],
                                    "diagnostic_score": job["diagnostic_score"],
                                    "historical_val_loss_available": "val_loss" in history.columns,
                                    "loss_batch_size": job["loss_batch_size"],
                                    "loss_measurement_seed": job["loss_measurement_seed"]}}
    save_table(pd.DataFrame(flatten_settings(settings)), out / "experiment_settings.csv")
    save_table(history, out / "training_history.csv")
    save_table(pd.DataFrame([context["result"]]), out / "run_summary.csv")
    report = evaluate_records(records, context["classes"], detailed=True)
    summary = {**report["summary"], "original_best_val_map_75_95": context["result"]["best_val_map_75_95"]}
    summary["difference_from_original_best"] = summary["map_75_95"] - summary["original_best_val_map_75_95"]
    if abs(summary["difference_from_original_best"]) > 1e-5:
        raise ValueError("저장된 best mAP와 재계산 mAP가 다릅니다. 같은 예측·패키지 버전인지 확인하세요. 원본 점수를 덮어쓰지 않습니다.")
    save_table(pd.DataFrame([summary]), out / "map_summary.csv")
    save_table(report["classes"], out / "class_ap.csv")
    save_table(report["ious"], out / "iou_ap.csv")
    np.savez_compressed(out / "precision_recall.npz", precision=report["precision"], category_ids=report["category_ids"])
    matrix, events = detection_errors(records, context["classes"]["category_id"],
                                      iou_threshold=job["diagnostic_iou"], score_threshold=job["diagnostic_score"])
    save_table(matrix, out / "confusion_counts.csv", index=True)
    normalized = matrix.div(matrix.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    save_table(normalized, out / "confusion_row_normalized.csv", index=True)
    save_table(events, out / "error_events.csv")
    per_class, pairs = diagnostic_tables(matrix, context["classes"])
    save_table(per_class, out / "class_error_summary.csv")
    save_table(pairs, out / "confused_pairs.csv")
    kinds = ["correct", "class_confusion", "localization_or_missed", "missed", "duplicate", "extra"]
    save_table(events["kind"].value_counts().reindex(kinds, fill_value=0).rename_axis("kind").reset_index(name="events"),
               out / "error_type_summary.csv")
    save_figure(plot_learning_curves([context["run"]]), out / "learning_curves.png")
    save_figure(plot_ap(report), out / "class_iou_ap.png")
    save_figure(plot_confusion(matrix, normalize=False), out / "confusion_counts.png")
    save_figure(plot_confusion(matrix, normalize=True), out / "confusion_normalized.png")
    if bundle is not None:
        save_figure(preview_records(records, events, count=job["preview_count"], score_threshold=job["diagnostic_score"]),
                    out / "error_examples.png")
    original_hashes = {p.name: file_sha256(p) for p in context["run"].iterdir()
                       if p.is_file() and p.suffix in {".json", ".csv", ".pt"}}
    write_json(out / "analysis_manifest.json", {"source_file_sha256": original_hashes,
               "original_signature": context["manifest"]["signature"], "analysis_environment": measurement_environment(),
               "settings": settings["analysis_policy"], "source_images_verified": bundle is not None})
    notes = """# Seed 0 분석 자료의 해석 범위

- mAP는 bbox IoU 0.75, 0.80, 0.85, 0.90, 0.95의 평균이며 maxDets=100이다.
- 혼동행렬·precision·recall·F1은 diagnostic_settings의 고정 IoU·score 기준이다. mAP와 다른 기준이다.
- class_confusion은 위치 대응이 된 오분류, localization_or_missed는 같은 클래스의 낮은 IoU 후보가 있는 미검출이다. 위치 오류의 확정 판정은 아니다.
- 오류 event는 정답과 예측 관점이 섞여 있어 합계가 정답 객체 수와 같지 않을 수 있다. 중복·추가 예측도 별도 event다.
- AP가 없거나 정답 수가 0인 클래스는 성능 0으로 단정하지 않는다. 클래스 빈도와 오류율을 함께 확인한다.
- 과거 epoch별 val loss가 없다면 그 곡선은 복원하지 않는다. best·last loss는 현재 환경에서 추가 측정한 두 시점이다.
- train loss와 val mAP 흐름은 과적합의 단서를 제공한다. 최고 검증 epoch는 이 실행 내 최고 mAP 시점이며, 과적합을 자동 확정하지 않는다.
- 학습 중 train loss는 batch 16의 학습 경로 기록이다. 사후 loss는 별도 소배치·고정 seed 측정이므로 두 값을 직접 동일 조건으로 간주하지 않는다.
- checkpoint_losses.csv에서 같은 프로토콜로 측정한 train·val만 서로 직접 비교한다. 두 checkpoint가 같은 epoch면 독립적인 두 시점이 아니다.
- 원래 학습 환경과 현재 분석 환경은 experiment_settings.csv에 구분해 기록한다. 예측 캐시를 읽을 때 원본 점수와 재계산 점수도 대조한다.
"""
    (out / "READ_ME.md").write_text(notes, encoding="utf-8")
    write_json(out / "diagnostic_settings.json", {"iou": job["diagnostic_iou"], "score": job["diagnostic_score"],
                                                  "rows": "true", "columns": "predicted"})
    print("저장된 결과 분석 완료:", out, flush=True)
    print(pd.DataFrame([summary]).to_string(index=False), flush=True)


def freeze_normalization(module):
    """원래 Faster R-CNN과 동일한 FrozenBN eps=0 구조를 복원한다.

    Args:
        module (Module): 변환할 모델 또는 하위 모듈.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, torch.nn.BatchNorm2d):
            frozen = FrozenBatchNorm2d(child.num_features, eps=0.0)
            for attr in ["weight", "bias", "running_mean", "running_var"]:
                getattr(frozen, attr).copy_(getattr(child, attr).detach())
            setattr(module, name, frozen)
        elif isinstance(child, FrozenBatchNorm2d):
            child.eps = 0.0
        else:
            freeze_normalization(child)


def restore_checkpoint(context, kind, device):
    """COCO 재다운로드 없이 원래 구조와 전체 모델 가중치를 엄격하게 복원한다.

    Args:
        context (dict): 원래 실행 설정.
        kind (str): best 또는 last.
        device (device): 측정 장치.

    Returns:
        tuple: 복원 모델, 저장 epoch, 파일 해시.
    """
    config = context["config"]
    if config["normalization"] != "FrozenBatchNorm2d_eps0_fixed_statistics_and_affine" or config["precision"] != "fp32":
        raise ValueError("원래 정규화·정밀도 정책을 지원하지 않습니다. config.json을 확인하세요.")
    checkpoint_path = context["run"] / f"{kind}.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if checkpoint["signature"] != context["manifest"]["signature"]:
        raise ValueError(f"{kind}.pt가 다른 실행의 체크포인트입니다. 같은 실행 폴더의 파일을 준비하세요.")
    expected_epoch = context["result"]["best_epoch" if kind == "best" else "last_epoch"]
    if checkpoint["epoch"] != expected_epoch:
        raise ValueError(f"{kind}.pt의 epoch가 result.json과 다릅니다. 원본 파일을 확인하세요.")
    model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None,
                                    min_size=config["image_size"], max_size=config["image_size"])
    freeze_normalization(model)
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, len(context["classes"]) + 1)
    model.roi_heads.score_thresh = config["score_threshold"]
    model.roi_heads.nms_thresh = config["nms_iou"]
    model.roi_heads.detections_per_img = config["max_detections"]
    model.load_state_dict(checkpoint["model"], strict=True)
    epoch = int(checkpoint["epoch"])
    del checkpoint
    model.to(device).eval()
    return model, epoch, file_sha256(checkpoint_path)


def model_state_digest(model):
    """파라미터와 모든 영속 버퍼의 내용 해시를 계산한다.

    Args:
        model (Module): 검사 대상 모델.

    Returns:
        str: 모델 상태 식별자.
    """
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        digest.update(name.encode())
        digest.update(str((tensor.dtype, tuple(tensor.shape))).encode())
        digest.update(tensor.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()


@contextlib.contextmanager
def loss_measurement_context(model, seed, deterministic=True):
    """손실 경로만 활성화하고 모드·난수 상태를 원래대로 복구한다.

    Args:
        model (Module): 복원한 Faster R-CNN.
        seed (int): RPN·RoI 표본 추출의 고정 seed.
        deterministic (bool): 원래 설정의 결정적 연산 사용 여부.
    """
    modes = {module: module.training for module in model.modules()}
    python_rng, numpy_rng = random.getstate(), np.random.get_state()
    backend_flags = (torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic,
                     torch.are_deterministic_algorithms_enabled(), torch.is_deterministic_algorithms_warn_only_enabled())
    device = next(model.parameters()).device
    devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == "cuda" else []
    try:
        with torch.random.fork_rng(devices=devices):
            random.seed(seed)
            np.random.seed(seed)
            torch.random.default_generator.manual_seed(seed)
            if device.type == "cuda":
                with torch.cuda.device(device):
                    torch.cuda.manual_seed(seed)
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = deterministic
            torch.use_deterministic_algorithms(deterministic)
            model.train()
            for module in model.modules():
                if isinstance(module, (torch.nn.modules.batchnorm._BatchNorm, torch.nn.modules.dropout._DropoutNd)):
                    module.eval()
            with torch.no_grad():
                yield
    finally:
        for module, training in modes.items():
            module.training = training
        random.setstate(python_rng)
        np.random.set_state(numpy_rng)
        torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic = backend_flags[:2]
        torch.use_deterministic_algorithms(backend_flags[2], warn_only=backend_flags[3])


def measure_checkpoint_loss(model, dataset, batch_size, seed, progress_every=20, deterministic=True):
    """가중치·버퍼를 갱신하지 않고 고정 데이터 전체의 네 검출 손실을 측정한다.

    Args:
        model (Module): 복원 모델.
        dataset (Dataset): 고정 순서 데이터.
        batch_size (int): 사후 측정 배치 크기.
        seed (int): 표본 추출 seed.
        progress_every (int): 진행 출력 간격.
        deterministic (bool): 원래 설정의 결정적 연산 사용 여부.

    Returns:
        dict: 이미지 수, 전체·세부 손실, 상태 불변 여부.
    """
    if batch_size < 1 or len(dataset) < 1:
        raise ValueError("양수 batch와 비어 있지 않은 데이터가 필요합니다. 설정·고정 분할을 확인하세요.")
    device = next(model.parameters()).device
    before = model_state_digest(model)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0,
                        collate_fn=collate_detection, generator=torch.Generator().manual_seed(seed))
    totals, seen = Counter(), 0
    expected = {"loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"}
    started = time.perf_counter()
    with loss_measurement_context(model, seed, deterministic):
        for step, (images, targets) in enumerate(loader, 1):
            values = model([image.to(device) for image in images],
                           [{k: v.to(device) for k, v in target.items()} for target in targets])
            if not isinstance(values, dict) or set(values) != expected:
                raise ValueError("Faster R-CNN의 네 손실이 반환되지 않았습니다. 모델 구조·torchvision 버전을 확인하세요.")
            for name, value in values.items():
                if value.numel() != 1 or not torch.isfinite(value).all():
                    raise FloatingPointError("손실이 유한한 스칼라가 아닙니다. 라벨·입력·모델을 확인하세요.")
                totals[name] += float(value) * len(images)
            seen += len(images)
            if step == 1 or step % progress_every == 0 or seen == len(dataset):
                print(f"손실 측정 {seen}/{len(dataset)} images | {time.perf_counter()-started:.1f}s", flush=True)
    if model_state_digest(model) != before:
        raise RuntimeError("손실 측정 중 모델 상태가 바뀌었습니다. 결과를 사용하지 말고 코드를 점검하세요.")
    if seen != len(dataset):
        raise RuntimeError("데이터 전체를 측정하지 못했습니다. 해당 측정을 다시 실행하세요.")
    components = {key: value / seen for key, value in totals.items()}
    return {"loss_total": sum(components.values()), **components, "images": seen,
            "batch_size": batch_size, "measurement_seed": seed, "state_unchanged": True,
            "seconds": time.perf_counter() - started}


def loss_device(requested):
    """팀 CUDA 버전을 유지한 채 실행 가능한 장치인지 확인한다.

    Args:
        requested (str): cuda 또는 cpu.

    Returns:
        device: 사용할 장치.
    """
    if requested == "cpu":
        return torch.device("cpu")
    if requested != "cuda":
        raise ValueError("LOSS_DEVICE는 cuda 또는 cpu여야 합니다. 셀 1을 수정하세요.")
    if not torch.cuda.is_available():
        raise RuntimeError("팀 잠금 환경에서 CUDA를 사용할 수 없습니다. GPU 선택·드라이버 보고서를 확인하세요. CPU 저장 결과 분석은 이미 가능하며 패키지를 임의로 낮추지 않습니다.")
    # 실제 연산과 torchvision 확장도 검사한다. GPU 선택만으로 호환성을 가정하지 않는다.
    device = torch.device("cuda:0")
    boxes = torch.tensor([[0., 0., 2., 2.], [0., 0., 2., 2.]], device=device)
    if nms(boxes, torch.tensor([0.9, 0.8], device=device), 0.5).tolist() != [0]:
        raise RuntimeError("GPU NMS 검증에 실패했습니다. CUDA·torchvision 환경을 확인하세요.")
    return device


def checkpoint_loss_reports(job):
    """best·last의 사후 train·val loss를 동일 조건으로 측정하고 원본과 별도로 저장한다.

    Args:
        job (dict): 입력 자료와 측정 프로토콜.
    """
    context = load_run_context(job)
    bundle = verified_dataset(context, job)
    # 캐시와 best checkpoint가 같은 검증 데이터에 속하는지도 재검증한다.
    load_cached_records(context, bundle)
    device = loss_device(job["loss_device"])
    splits = job["loss_splits"]
    if not splits or len(set(splits)) != len(splits) or not set(splits).issubset({"train", "val"}):
        raise ValueError("LOSS_SPLITS는 train·val 중 중복 없는 목록이어야 합니다. 셀 1을 확인하세요.")
    protocol = {"batch_size": job["loss_batch_size"], "seed": job["loss_measurement_seed"],
                "mode": "train_path_frozen_norm_no_grad", "dtype": "float32", "augmentation": "none",
                "deterministic_algorithms": context["config"].get("deterministic", True),
                "data_identity": bundle["identity"], "original_signature": context["manifest"]["signature"],
                "measurement_environment": measurement_environment()}
    protocol_id = stable_hash(protocol)
    rows = []
    for kind in ["best", "last"]:
        checkpoint_hash = file_sha256(context["run"] / f"{kind}.pt")
        cache = context["out"] / f"{kind}_posthoc_loss.json"
        existing = json.loads(cache.read_text()) if cache.is_file() else None
        if (existing and existing["protocol_id"] == protocol_id and existing["checkpoint_sha256"] == checkpoint_hash
                and {r["split"] for r in existing["rows"]} == set(splits)):
            print(f"동일 조건의 {kind} 손실 재사용", flush=True)
            rows.extend(existing["rows"])
            continue
        # 모델 초기화도 호출자의 난수 상태에 영향을 남기지 않는다.
        with torch.random.fork_rng(devices=[]):
            model, epoch, checkpoint_hash = restore_checkpoint(context, kind, device)
        current_rows = []
        try:
            for split in splits:
                print(f"{kind} | epoch {epoch} | {split} | 가중치 갱신 없이 측정", flush=True)
                values = measure_checkpoint_loss(model, make_dataset(bundle, split),
                                                 job["loss_batch_size"], job["loss_measurement_seed"],
                                                 deterministic=context["config"].get("deterministic", True))
                row = {"checkpoint": kind, "epoch": epoch, "split": split, "posthoc_measurement": True,
                       "checkpoint_sha256": checkpoint_hash, "protocol_id": protocol_id, **values}
                current_rows.append(row)
        finally:
            del model
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
        write_json(cache, {"protocol_id": protocol_id, "checkpoint_sha256": checkpoint_hash,
                           "protocol": protocol, "rows": current_rows})
        rows.extend(current_rows)
        save_table(pd.DataFrame(rows), context["out"] / "checkpoint_losses.csv")
    frame = pd.DataFrame(rows)
    save_table(frame, context["out"] / "checkpoint_losses.csv")
    write_json(context["out"] / "loss_protocol.json", protocol)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for split in splits:
        selected = frame.loc[frame["split"].eq(split)]
        axes[0].scatter(selected["epoch"], selected["loss_total"], label=f"{split}: post-hoc checkpoints", s=65)
        for row in selected.itertuples():
            axes[0].annotate(row.checkpoint, (row.epoch, row.loss_total), xytext=(4, 5), textcoords="offset points")
    axes[0].set(title="Checkpoint loss: measured now, not an epoch curve", xlabel="Saved epoch", ylabel="Loss")
    axes[0].legend(fontsize=8)
    components = ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]
    frame.set_index(["checkpoint", "split"])[components].plot.bar(ax=axes[1], stacked=True)
    axes[1].set(title="Checkpoint loss components", ylabel="Loss", xlabel="Checkpoint / split")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    save_figure(fig, context["out"] / "checkpoint_losses.png")
    print(frame[["checkpoint", "epoch", "split", "loss_total", "images", "state_unchanged"]].to_string(index=False), flush=True)


def dispatch_analysis(job):
    """지정된 분석 단계만 실행한다.

    Args:
        job (dict): stage와 각 입력의 설정.
    """
    if job["stage"] == "saved":
        saved_result_reports(job)
    elif job["stage"] == "loss":
        checkpoint_loss_reports(job)
    else:
        raise ValueError("지원하지 않는 분석 단계입니다. saved 또는 loss를 지정하세요.")

'''

INLINE_ANALYSIS_BLOCKS['environment_probe'] = r'''
"""분석용 자식 환경이 팀 잠금 버전과 일치하는지 검사한다."""

import hashlib
import importlib.metadata as metadata
import json
import subprocess
import sys
import tomllib
from pathlib import Path

from packaging.markers import Marker
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from packaging.version import Version


def verify_team_environment(reference_root, original_manifest, report_path=None):
    """OS별 잠금 버전·원래 학습 패키지·드라이버 정보를 확인한다.

    Args:
        reference_root (str | Path): 내장 또는 업로드한 팀 환경 파일 경로.
        original_manifest (dict): 원래 학습 config.json.
        report_path (str | Path | None): 환경 보고서 출력 경로.

    Returns:
        dict: 잠금 일치 여부와 GPU 사용 가능 여부.
    """
    root = Path(reference_root)
    expected_hash = original_manifest["recipe"]["runtime"]["uv_lock_sha256"]
    if hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest() != expected_hash:
        raise RuntimeError("잠금 파일이 원래 학습과 다릅니다. 환경 설정 셀을 다시 확인하세요.")
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("분석 자식 프로세스가 Python 3.12가 아닙니다. 환경 설정 셀부터 다시 실행하세요.")
    project = tomllib.loads((root / "pyproject.toml").read_text())
    lock = tomllib.loads((root / "uv.lock").read_text())
    required = {canonicalize_name(Requirement(item).name) for item in project["project"]["dependencies"]} | {"pandas"}
    rows = []
    for name in sorted(required):
        versions = {p["version"] for p in lock["package"] if canonicalize_name(p["name"]) == name
                    and "version" in p and (not p.get("resolution-markers")
                    or any(Marker(marker).evaluate() for marker in p["resolution-markers"]))}
        if len(versions) != 1:
            raise RuntimeError(f"{name}의 OS별 잠금 버전을 결정할 수 없습니다. 환경 파일을 확인하세요.")
        locked, installed = next(iter(versions)), metadata.version(name)
        if Version(locked) != Version(installed):
            raise RuntimeError(f"{name} 버전이 다릅니다: {installed}, 잠금 {locked}. 환경 설치 셀을 다시 실행하세요.")
        rows.append({"package": name, "locked": locked, "installed": installed})
    runtime = original_manifest["recipe"]["runtime"]
    for name, original in runtime["packages"].items():
        if Version(metadata.version(name)) != Version(original):
            raise RuntimeError(f"원래 학습과 {name} 버전이 다릅니다. 같은 잠금 환경을 준비하세요.")
    import torch
    import torchvision
    from torchvision.ops import nms
    # GPU가 막혀 있어도 CPU 저장 자료 분석은 수행할 수 있다.
    cpu_keep = nms(torch.tensor([[0., 0., 2., 2.], [0., 0., 2., 2.]]), torch.tensor([0.9, 0.8]), 0.5)
    if cpu_keep.tolist() != [0]:
        raise RuntimeError("torchvision CPU NMS가 동작하지 않습니다. 설치 환경을 확인하세요.")
    drivers = []
    gpu_ok = False
    problem = None
    try:
        command = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                                 capture_output=True, text=True)
        if command.returncode == 0:
            drivers = command.stdout.strip().splitlines()
        if not torch.cuda.is_available():
            raise RuntimeError("팀 잠금 PyTorch에서 CUDA가 사용 불가합니다. GPU 선택·드라이버를 확인하세요.")
        driver_numbers = [int(line.rsplit(",", 1)[-1].strip().split(".")[0]) for line in drivers]
        if torch.version.cuda and int(torch.version.cuda.split(".")[0]) >= 13 and driver_numbers and min(driver_numbers) < 580:
            raise RuntimeError("팀 CUDA 13 빌드는 NVIDIA 드라이버 580 이상이 필요합니다. Kaggle 드라이버는 노트북에서 교체하지 않습니다.")
        boxes = torch.tensor([[0., 0., 2., 2.], [0., 0., 2., 2.]], device="cuda:0")
        if nms(boxes, torch.tensor([0.9, 0.8], device="cuda:0"), 0.5).tolist() != [0]:
            raise RuntimeError("GPU NMS 결과가 예상과 다릅니다.")
        gpu_ok = True
    except Exception as exc:
        problem = str(exc)
    report = {"python": sys.version.split()[0], "packages": rows, "uv_lock_sha256": expected_hash,
              "torch_cuda": torch.version.cuda, "gpu_driver_report": drivers,
              "gpu_usable_with_team_versions": gpu_ok, "gpu_problem": problem,
              "cpu_artifact_analysis_available": True}
    if report_path:
        Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

'''

print("내장 함수 준비 완료. 다음 셀 4를 실행하세요.")


### 셀 4 · 분석 실행 도구·패키지·GPU 검증

현재 학습 기록의 패키지 버전과 별도 환경을 대조한다. GPU가 막혀 있어도 CPU로 수행할 저장 자료 분석은 계속 가능하다.
GPU 사용 불가 메시지가 있으면 임의로 패키지 버전을 낮추지 말고 원인을 확인한다.
CUDA 13 빌드는 일반적으로 드라이버 580 이상이 필요하며, 이 노트북은 Kaggle 시스템 드라이버를 바꾸지 않는다.


In [ ]:
import csv
import os
import subprocess
from IPython.display import Image as DisplayImage, FileLink, display

RUNTIME_ROOT = ENV_ROOT / "runtime"
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
BLOCK_FILE = RUNTIME_ROOT / "inline_functions.json"
BLOCK_FILE.write_text(json.dumps(INLINE_ANALYSIS_BLOCKS, ensure_ascii=False), encoding="utf-8")
WORKER_FILE = RUNTIME_ROOT / "analysis_worker.py"
WORKER_FILE.write_text(
    "import json,sys\nfrom pathlib import Path\n"
    "job=json.loads(Path(sys.argv[1]).read_text())\n"
    "blocks=json.loads(Path(job['block_file']).read_text())\n"
    "scope={'__name__':'kaggle_seed0_analysis'}\n"
    "exec(compile(blocks['environment_probe'],'<environment_probe>','exec'),scope)\n"
    "manifest=json.loads(Path(job['run_dir'],'config.json').read_text())\n"
    "report=scope['verify_team_environment'](job['reference_root'],manifest,job['environment_report'])\n"
    "if job['stage']=='probe':\n"
    " print(json.dumps(report,ensure_ascii=False,indent=2),flush=True)\n"
    "else:\n"
    " for name in ['helpers','data','analysis','reports_and_loss']:\n"
    "  exec(compile(blocks[name],'<'+name+'>','exec'),scope)\n"
    " scope['dispatch_analysis'](job)\n", encoding="utf-8")


def run_analysis_stage(stage):
    """Kaggle 커널을 바꾸지 않고 팀 Python으로 분석을 수행한다.

    Args:
        stage (str): probe, saved, loss 중 실행 단계.
    """
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    job = {"stage": stage, "run_dir": str(RUN_DIR), "output_dir": str(OUTPUT_DIR),
           "block_file": str(BLOCK_FILE), "reference_root": str(REFERENCE_ROOT),
           "environment_report": str(OUTPUT_DIR / "environment_report.json"),
           "image_dir": str(IMAGE_DIR) if IMAGE_DIR else None,
           "annotation_dir": str(ANNOTATION_DIR) if ANNOTATION_DIR else None,
           "split_path": str(SPLIT_PATH) if SPLIT_PATH else None,
           "fixes_path": str(FIXES_PATH) if FIXES_PATH else None,
           "loss_device": LOSS_DEVICE, "loss_batch_size": LOSS_BATCH_SIZE,
           "loss_measurement_seed": LOSS_MEASUREMENT_SEED, "loss_splits": LOSS_SPLITS,
           "diagnostic_iou": DIAGNOSTIC_IOU, "diagnostic_score": DIAGNOSTIC_SCORE,
           "preview_count": PREVIEW_COUNT}
    path = RUNTIME_ROOT / f"job_{stage}.json"
    path.write_text(json.dumps(job, ensure_ascii=False), encoding="utf-8")
    child_env = {**os.environ, "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1",
                 "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "OMP_NUM_THREADS": "2", "PYTHONNOUSERSITE": "1"}
    # Kaggle 기본 커널의 패키지 경로가 분석 환경에 섞이지 않게 한다.
    child_env.pop("PYTHONPATH", None)
    stream_command([str(ANALYSIS_PYTHON), "-u", str(WORKER_FILE), str(path)], env=child_env)


def show_csv(name, limit=15):
    """CSV 결과의 앞부분을 Kaggle 기본 커널에서도 표시한다.

    Args:
        name (str): 결과 CSV 이름.
        limit (int): 화면에 표시할 최대 행 수.
    """
    path = OUTPUT_DIR / name
    if not path.is_file():
        print("아직 생성되지 않음:", name)
        return
    import html
    from IPython.display import HTML
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.reader(stream)
        rows = [row for _, row in zip(range(limit + 1), reader)]
    body = "".join("<tr>" + "".join("<td>" + html.escape(value) + "</td>" for value in row) + "</tr>" for row in rows)
    display(HTML("<p>" + html.escape(name) + "</p><table>" + body + "</table>"))


run_analysis_stage("probe")


## 3. 저장 결과의 평가·오류 분석

### 셀 5 · 원본 파일 구성 확인

학습 폴더 전체가 같은 실행의 것인지 확인한다. 원본은 읽기만 하며 모든 새 출력은 별도 폴더에 저장한다.
`last.pt`는 원래 학습 완료 지점의 모델이다. 새 seed로 이어 학습하거나 가중치를 갱신하지 않는다.


In [ ]:
required_files = ["config.json", "result.json", "metrics.csv", "class_map.csv", "validation_predictions.pt", "best.pt", "last.pt"]
missing = []
for name in required_files:
    path = RUN_DIR / name
    print(name, "|", f"{path.stat().st_size / 1024**2:.2f} MiB" if path.is_file() else "없음")
    if not path.is_file():
        missing.append(name)
if missing:
    raise FileNotFoundError(f"누락 파일: {missing}. Drive의 동일 실행 폴더에서 보완해 주세요.")
print("기존 실행:", SOURCE_MANIFEST["run_name"], "| seed:", SOURCE_MANIFEST["spec"]["seed"])


### 셀 6 · 설정 CSV·학습 곡선·mAP·혼동행렬·오류 요약 생성

- 학습 설정·원래 환경·분석 환경·진단 조건을 `experiment_settings.csv`로 정리한다.
- `metrics.csv`의 train loss·val mAP 곡선과 최고 epoch를 정리한다. 과거 val loss가 없는 사실도 기록한다.
- 저장된 검증 정답·예측으로 mAP를 재계산하고 원래 최고 점수와 일치하는지 확인한다.
- 전체·클래스별·IoU별 AP, 혼동행렬 원시 건수·행 정규화, 클래스별 오류율·진단 precision/recall/F1,
  자주 혼동한 클래스 쌍, 오류 유형별 건수를 저장한다.
- 진단 IoU·score는 mAP 기준과 별개다. `localization_or_missed`는 위치 오류의 의심 후보이며 확정 판정이 아니다.
- 원본 이미지가 연결돼 있으면 고정 분할·라벨·이미지 해시와 예측 캐시를 검증하고 오류 이미지를 다시 그린다.
  이미지가 없으면 수치 분석만 수행하고 원본 이미지·체크포인트와의 완전한 대조가 미완료임을 명시한다.


In [ ]:
run_analysis_stage("saved")
for name in ["run_summary.csv", "map_summary.csv", "class_error_summary.csv", "confused_pairs.csv", "error_type_summary.csv"]:
    show_csv(name)
for name in ["learning_curves.png", "class_iou_ap.png", "confusion_normalized.png", "error_examples.png"]:
    path = OUTPUT_DIR / name
    if path.is_file():
        display(DisplayImage(filename=str(path), width=1000))
print("전체 설정 CSV:", OUTPUT_DIR / "experiment_settings.csv")


## 4. Best·Last 손실 추가 측정

### 셀 7 · seed 0 체크포인트의 train·validation loss

**가중치 갱신·역전파·optimizer 실행은 없다.** 모델의 손실 반환 경로만 켜고 정규화·Dropout을 고정한다.
Python·NumPy·PyTorch 난수와 모델 모드를 복구하고, 측정 전후 파라미터·버퍼의 내용 해시를 대조한다.
같은 batch·표본 추출 seed를 best·last 모두에 사용한다. RPN·RoI의 표본 추출을 포함하는 사후 측정이다.

원래 train loss 곡선과 **별도 CSV·별도 그래프**로 저장한다. 두 점을 과거 val loss 곡선처럼 연결하지 않는다.
네 손실(classifier, box regression, RPN objectness, RPN box regression)도 저장한다.
측정 완료한 동일 조건은 다음 실행에서 재사용한다. 이미지·GPU·환경 또는 측정 조건이 다르면 재측정한다.

GPU 검증이 실패한 경우에는 이 셀을 실행하지 말고 `environment_report.json`의 원인을 확인한다.
CPU 측정을 명시적으로 원하면 셀 1의 `LOSS_DEVICE="cpu"`로 바꿀 수 있지만 시간이 오래 걸릴 수 있다.


In [ ]:
run_analysis_stage("loss")
show_csv("checkpoint_losses.csv")
display(DisplayImage(filename=str(OUTPUT_DIR / "checkpoint_losses.png"), width=1100))


## 5. 결과 저장·다운로드

### 셀 8 · 분석 결과 목록·ZIP 저장

생성된 CSV·그래프·환경·측정 조건만 압축한다. 입력 데이터·체크포인트·API 키는 넣지 않는다.
아래 ZIP을 다운로드한다. Kaggle 세션 종료 전 Save Version에서 출력 보존 여부도 확인한다.
손실 측정이 불가능했더라도 셀 6까지 생성된 결과는 다운로드할 수 있다.


In [ ]:
import zipfile

if not (OUTPUT_DIR / "analysis_manifest.json").is_file():
    raise RuntimeError("저장 결과 분석이 완료되지 않았습니다. 셀 6을 먼저 실행하세요.")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(path.name, f"({path.stat().st_size / 1024:.1f} KiB)")
ZIP_PATH = OUTPUT_ROOT / (OUTPUT_DIR.name + ".zip")
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".csv", ".json", ".png", ".md", ".npz"}:
            archive.write(path, arcname=path.relative_to(OUTPUT_DIR))
print("분석 결과 ZIP:", ZIP_PATH)
display(FileLink(str(ZIP_PATH)))
